<a href="https://colab.research.google.com/github/sabahoth01/NLP-courses-SPBU/blob/task2/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install torch
# !pip install torchvision
# !pip install torchaudio
# !pip install numpy
# !pip install matplotlib
# !pip install graphviz
# !pip install notebook
# !pip install -U datasets
# !pip install transformers
# !pip install tqdm

In [ ]:
# import torch
# import torch.nn as nn
# from torch.nn import functional as F
# from datasets import load_dataset
# from tqdm import tqdm
# import math

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from tqdm import tqdm

# Dataset preparation
try:
    from datasets import load_dataset
except ImportError:
    print("Install 'datasets' with: pip install datasets")

# Data Preparation (same as original)
def prepare_russian_data():
    print("Loading Russian dataset...")
    try:
        dataset = load_dataset("sberquad")
        texts = []
        for split in dataset.keys():
            for example in tqdm(dataset[split], desc=f"Processing {split}"):
                text = f"{example['question']} {example['context']}".strip()
                if len(text) > 50:
                    texts.append(text)
        with open('russian_text.txt', 'w', encoding='utf-8') as f:
            f.write("\n".join(texts))
        print(f"Total samples: {len(texts)}")
        return texts
    except Exception as e:
        print(f"Error loading dataset: {e}")
        sample_text = "Русский язык является одним из самых распространенных языков в мире..."
        with open('russian_text.txt', 'w', encoding='utf-8') as f:
            f.write(sample_text)
        return [sample_text]

prepare_russian_data()

with open('russian_text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Character mapping
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Data
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

# Hyperparameters
batch_size = 16
block_size = 32
max_iters = 4500  # Reduced for faster run
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

torch.manual_seed(1337)

# Data loading
def get_batch(split):
    data_ = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_) - block_size, (batch_size,))
    x = torch.stack([data_[i:i+block_size] for i in ix])
    y = torch.stack([data_[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

Loading Russian dataset...


Processing test: 100%|██████████| 23936/23936 [00:01<00:00, 14258.17it/s]


Total samples: 74300


In [ ]:
# RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.norm(2, dim=-1, keepdim=True) * (1.0 / (x.shape[-1] ** 0.5))
        return self.scale * x / (norm + self.eps)

# Mixture of Experts (MoE)
class MoE(nn.Module):
    def __init__(self, n_embd, num_experts=4):
        super().__init__()
        self.experts = nn.ModuleList([nn.Linear(n_embd, n_embd) for _ in range(num_experts)])
        self.gate = nn.Linear(n_embd, num_experts)
    def forward(self, x):
        gate_scores = F.softmax(self.gate(x), dim=-1)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=-1)
        return (expert_outputs * gate_scores.unsqueeze(-2)).sum(dim=-1)


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.moe = MoE(n_embd)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.relu(self.moe(x)))

# class Block(nn.Module):
#     def __init__(self, n_embd, n_head):
#         super().__init__()
#         head_size = n_embd // n_head
#         self.sa = MultiHeadAttention(n_head, head_size)
#         self.ffwd = FeedFoward(n_embd)
#         self.ln1 = RMSNorm(n_embd)
#         self.ln2 = RMSNorm(n_embd)
#     def forward(self, x):
#         x = x + self.sa(self.ln1(x))
#         x = x + self.ffwd(self.ln2(x))
#         return x
class Block(nn.Module):
    def __init__(self, n_embd, n_head, norm_layer=nn.LayerNorm, ffn_layer=None):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ln1 = norm_layer(n_embd)
        self.ln2 = norm_layer(n_embd)
        if ffn_layer is not None:
            self.ffwd = ffn_layer(n_embd)
        else:
            self.ffwd = nn.Sequential(
                nn.Linear(n_embd, 4 * n_embd),
                nn.ReLU(),
                nn.Linear(4 * n_embd, n_embd),
                nn.Dropout(dropout)
            )
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


# Bigram Model
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, norm_layer=RMSNorm, ffn_layer=lambda d: MoE(d)) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
# super simple bigram model
class BigramLanguageModel_lNorm(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, norm_layer=nn.LayerNorm) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

# Simple Unigram Model
class UnigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.embedding(idx).mean(dim=1)
        if targets is not None:
            loss = F.cross_entropy(logits, targets[:, 0])
        else:
            loss = None
        return logits.unsqueeze(1), loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# New GPT Model (like GPT-2)
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, norm_layer=nn.LayerNorm) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)  # GPT often uses LayerNorm
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

class GPT_RMSMoE(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, norm_layer=RMSNorm, ffn_layer=lambda d: MoE(d)) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1)) if targets is not None else None
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

class BigramLanguageModel_RMSNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.norm = RMSNorm(n_embd)

    def forward(self, idx, targets=None):
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(idx.size(1), device=device))
        x = tok_emb + pos_emb
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])  # focus on last block_size tokens
            logits = logits[:, -1, :]  # get last token logits
            probs = F.softmax(logits, dim=-1)  # convert to probabilities
            idx_next = torch.multinomial(probs, num_samples=1)  # sample from distribution
            idx = torch.cat((idx, idx_next), dim=1)  # append sampled token
        return idx


# class BigramLanguageModel_MoE(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
#         self.position_embedding_table = nn.Embedding(block_size, n_embd)
#         self.moe = MoE(n_embd)
#         self.lm_head = nn.Linear(n_embd, vocab_size)

#     def forward(self, idx, targets=None):
#         tok_emb = self.token_embedding_table(idx)
#         pos_emb = self.position_embedding_table(torch.arange(idx.size(1), device=device))
#         x = tok_emb + pos_emb
#         x = self.moe(x)
#         logits = self.lm_head(x)
#         loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
#         return logits, loss

# class GPT_RMSNormOnly(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
#         self.position_embedding_table = nn.Embedding(block_size, n_embd)
#         # Wrap nn.Linear with correct in/out features
#         self.blocks = nn.Sequential(*[
#             Block(n_embd, n_head, norm_layer=RMSNorm, ffn_layer=lambda dim: nn.Sequential(
#                 nn.Linear(dim, 4*dim),
#                 nn.GELU(),
#                 nn.Linear(4*dim, dim)
#             )) for _ in range(n_layer)
#         ])
#         self.ln_f = RMSNorm(n_embd)
#         self.lm_head = nn.Linear(n_embd, vocab_size)

# class GPT_MoEOnly(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
#         self.position_embedding_table = nn.Embedding(block_size, n_embd)
#         self.blocks = nn.Sequential(*[
#             Block(n_embd, n_head, norm_layer=nn.LayerNorm, ffn_layer=lambda dim: MoE(dim))
#             for _ in range(n_layer)
#         ])
#         self.ln_f = nn.LayerNorm(n_embd)
#         self.lm_head = nn.Linear(n_embd, vocab_size)

class BigramLanguageModel_MoE(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.moe = MoE(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(idx.size(1), device=device))
        x = tok_emb + pos_emb
        x = self.moe(x)
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :]  # last token
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

class GPT_RMSNormOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, norm_layer=RMSNorm, ffn_layer=lambda dim: nn.Sequential(
                nn.Linear(dim, 4*dim),
                nn.GELU(),
                nn.Linear(4*dim, dim)
            )) for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(idx.size(1), device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

class GPT_MoEOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, norm_layer=nn.LayerNorm, ffn_layer=lambda dim: MoE(dim))
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(idx.size(1), device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



In [ ]:
# Train and Compare All
def train_and_compare():
    models = {
        # "Bigram+layerNorm": BigramLanguageModel_lNorm().to(device),
        # "Bigram+RMSNorm+MoE": BigramLanguageModel().to(device),
        # "Unigram": UnigramLanguageModel().to(device),
        # "GPT-layerNorm": GPTLanguageModel().to(device),
        # "GPT-RMSNorm+MoE": GPT_RMSMoE().to(device)
        "Unigram": UnigramLanguageModel().to(device),
        "Bigram+LayerNorm": BigramLanguageModel_lNorm().to(device),
        "Bigram+RMSNorm": BigramLanguageModel_RMSNorm().to(device),
        "Bigram+MoE": BigramLanguageModel_MoE().to(device),
        "Bigram+RMSNorm+MoE": BigramLanguageModel().to(device),
        "GPT-LayerNorm": GPTLanguageModel().to(device),
        "GPT-RMSNorm": GPT_RMSNormOnly().to(device),
        "GPT-MoE": GPT_MoEOnly().to(device),
        "GPT-RMSNorm+MoE": GPT_RMSMoE().to(device)
    }
    for name, model in models.items():
        print(f"\nTraining {name}...")
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
        for iter in range(max_iters):
            if iter % eval_interval == 0 or iter == max_iters - 1:
                losses = estimate_loss(model)
                print(f"{name} step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
            xb, yb = get_batch('train')
            logits, loss = model(xb, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        context = torch.zeros((1,1), dtype=torch.long, device=device)
        print(f"\n{name} generated text:")
        print(decode(model.generate(context, max_new_tokens=2000)[0].tolist()))

train_and_compare()



Training Unigram...
Unigram step 0: train loss 6.7330, val loss 6.7391
Unigram step 100: train loss 6.6039, val loss 6.6073
Unigram step 200: train loss 6.4846, val loss 6.4735
Unigram step 300: train loss 6.3605, val loss 6.3598
Unigram step 400: train loss 6.2288, val loss 6.2369
Unigram step 500: train loss 6.1101, val loss 6.1153
Unigram step 600: train loss 5.9956, val loss 6.0016
Unigram step 700: train loss 5.8822, val loss 5.8805
Unigram step 800: train loss 5.7649, val loss 5.7618
Unigram step 900: train loss 5.6504, val loss 5.6544
Unigram step 1000: train loss 5.5532, val loss 5.5398
Unigram step 1100: train loss 5.4348, val loss 5.4449
Unigram step 1200: train loss 5.3373, val loss 5.3535
Unigram step 1300: train loss 5.2321, val loss 5.2337
Unigram step 1400: train loss 5.1493, val loss 5.1414
Unigram step 1500: train loss 5.0486, val loss 5.0475
Unigram step 1600: train loss 4.9833, val loss 4.9698
Unigram step 1700: train loss 4.8797, val loss 4.8850
Unigram step 1800: 

Результаты показывают, что на этапе обучения модель **Bigram+LayerNorm** достигает наименьшей величины потерь как на тренировочной, так и на валидационной выборке. Другие модели (Unigram, Bigram+RMSNorm, Bigram+MoE и их комбинации) показывают худшие результаты по сравнению с **Bigram+LayerNorm**, а текст, сгенерированный этой моделью, более осмысленный. Это свидетельствует о том, что использование биграмм с нормализацией слоя (LayerNorm) значительно улучшает способность модели предсказывать последовательности.

**GPT-2+LayerNorm** Модель GPT работает лучше всего.
